# release16_nonconvex_checkpoint_objective_score_handoff_v1

Purpose: convert the expanded official replay candidate pool into a nonconvex checkpoint objective handoff. This is not an actor promotion. It scores the current official replay pool against a 30pp relative-return floor, a 20% MaxDD hard gate, and the sparse state-head slice precondition so the next run can train against the actual failure surface instead of expanding path-level thresholds again.

Governance:
- Baselines: canonical `soxlbacktest_qlib_walkforward_baseline_v1`; direct comparison `release16_expanded_official_candidate_pool_v1`; readiness reference `release16_nonconvex_checkpoint_objective_readiness_v1`.
- Role: `diagnose` / training-contract handoff.
- Target window: `wf_2020`, `wf_2022`, `wf_2023_2024` official replay evidence.
- Changed layer: `model_selection`, `checkpoint_objective`, `state_head_slice_guard`.
- External reference: converted SOXLBacktest qlib buy/sell baseline and release16 official replay candidate pool.
- Acceptance: no promotion unless all required windows have relative return >= 30pp, MaxDD < 20%, and sparse slice preconditions pass; actor retention remains disabled until an actual trained actor passes official three-fold replay.
- Historical success/failure: closest success is the converted SOXLBacktest qlib baseline used as reference; closest failure is `release16_expanded_official_candidate_pool_v1`, where the best runnable candidate still missed relative return and breached drawdown. This run is not a duplicate because it translates that pool into a training objective and handoff with explicit sparse-slice stop conditions.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "src" / "soxl_quant").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.training_objective_handoff import (
    derive_training_objective_handoff,
    render_training_objective_handoff,
)
EXPERIMENT = "release16_nonconvex_checkpoint_objective_score_handoff_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

EXPANDED_CARD = ROOT / "runtime" / "model_selection" / "card" / "release16_expanded_official_candidate_pool_v1"
READINESS_CARD = ROOT / "runtime" / "model_selection" / "card" / "release16_nonconvex_checkpoint_objective_readiness_v1"

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
DRAWDOWN_DRIFT_WEIGHT = 2.0
SPARSE_SLICE_GUARD_WEIGHT = 5.0
REQUIRED_WINDOWS = ["wf_2020", "wf_2022", "wf_2023_2024"]

SOURCE_PATHS = {
    "expanded_summary": EXPANDED_CARD / "summary.json",
    "expanded_candidate_windows": EXPANDED_CARD / "candidate_window_metrics_20dd.csv",
    "expanded_candidate_scores": EXPANDED_CARD / "official_candidate_scores_20dd.csv",
    "expanded_best_windows": EXPANDED_CARD / "best_candidate_window_gaps.csv",
    "readiness_summary": READINESS_CARD / "summary.json",
    "readiness_teacher": READINESS_CARD / "selected_fold_teacher_metrics.csv",
    "readiness_slice_precondition": READINESS_CARD / "state_head_slice_precondition.csv",
    "readiness_objective_recipe": READINESS_CARD / "objective_recipe.json",
}

missing = [str(path) for path in SOURCE_PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError(missing)

print("card", CARD)
print("sources", {name: str(path.relative_to(ROOT)) for name, path in SOURCE_PATHS.items()})


card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_nonconvex_checkpoint_objective_score_handoff_v1
sources {'expanded_summary': 'runtime/model_selection/card/release16_expanded_official_candidate_pool_v1/summary.json', 'expanded_candidate_windows': 'runtime/model_selection/card/release16_expanded_official_candidate_pool_v1/candidate_window_metrics_20dd.csv', 'expanded_candidate_scores': 'runtime/model_selection/card/release16_expanded_official_candidate_pool_v1/official_candidate_scores_20dd.csv', 'expanded_best_windows': 'runtime/model_selection/card/release16_expanded_official_candidate_pool_v1/best_candidate_window_gaps.csv', 'readiness_summary': 'runtime/model_selection/card/release16_nonconvex_checkpoint_objective_readiness_v1/summary.json', 'readiness_teacher': 'runtime/model_selection/card/release16_nonconvex_checkpoint_objective_readiness_v1/selected_fold_teacher_metrics.csv', 'readiness_slice_precondition': 'runt

In [2]:
def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if isinstance(value, tuple):
        return [json_safe(item) for item in value]
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        number = float(value)
        return None if not np.isfinite(number) else number
    if isinstance(value, float):
        return None if not np.isfinite(value) else value
    return value


expanded_summary = read_json(SOURCE_PATHS["expanded_summary"])
readiness_summary = read_json(SOURCE_PATHS["readiness_summary"])
readiness_recipe = read_json(SOURCE_PATHS["readiness_objective_recipe"])

candidate_windows = pd.read_csv(SOURCE_PATHS["expanded_candidate_windows"])
official_scores = pd.read_csv(SOURCE_PATHS["expanded_candidate_scores"])
best_windows = pd.read_csv(SOURCE_PATHS["expanded_best_windows"])
teacher_metrics = pd.read_csv(SOURCE_PATHS["readiness_teacher"])
slice_precondition = pd.read_csv(SOURCE_PATHS["readiness_slice_precondition"])

print({
    "official_candidate_windows": candidate_windows.shape,
    "official_scores": official_scores.shape,
    "best_windows": best_windows.shape,
    "teacher_metrics": teacher_metrics.shape,
    "slice_precondition": slice_precondition.shape,
})

display(official_scores.head(8))
display(slice_precondition)


{'official_candidate_windows': (123, 12), 'official_scores': (41, 19), 'best_windows': (3, 18), 'teacher_metrics': (3, 9), 'slice_precondition': (3, 5)}


,candidate_id,source_label,mechanism,candidate,window_count,gate_pass_count,full_gate,robust_gate_pass_count,robust_full_gate,near_miss,min_relative_return,max_max_dd,mean_avg_pos,max_relative_gap,max_dd_gap,min_relative_margin,min_max_dd_margin,total_gate_loss,failure_profile
0,release16_bull_reentry_participation_objective...,release16_bull_reentry_participation_objective...,official_daily_path_pool,bull65_hold3,3,1,False,1,False,False,14.302062,21.106156,0.236935,15.697938,1.106156,-15.697938,-1.106156,34.477859,joint_tradeoff
1,release16_bull_reentry_participation_objective...,release16_bull_reentry_participation_objective...,official_daily_path_pool,bull70_hold5,3,1,False,1,False,False,11.949062,20.939130,0.210516,18.050938,0.939130,-18.050938,-0.939130,43.140173,joint_tradeoff
2,release16_failedfold_overlay_checkpoint_screen...,release16_failedfold_overlay_checkpoint_screen_v1,official_daily_path_pool,bc_initialization,3,1,False,1,False,False,3.406380,16.652151,0.131609,26.593620,0.000000,-26.593620,3.347849,45.173038,trend_capture
3,release16_failedfold_overlay_checkpoint_screen...,release16_failedfold_overlay_checkpoint_screen_v1,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,3,1,False,1,False,False,3.406380,16.652151,0.131609,26.593620,0.000000,-26.593620,3.347849,45.173038,trend_capture
4,release16_rulebase_reversal_checkpoint_metric_...,release16_rulebase_reversal_checkpoint_metric_...,official_daily_path_pool,bc_initialization,3,1,False,1,False,False,5.082695,20.934612,0.174185,24.917305,0.934612,-24.917305,-0.934612,52.684889,joint_tradeoff
5,release16_rulebase_reversal_checkpoint_metric_...,release16_rulebase_reversal_checkpoint_metric_...,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,3,1,False,1,False,False,5.082695,20.934612,0.174185,24.917305,0.934612,-24.917305,-0.934612,52.684889,joint_tradeoff
6,release16_rulebase_reversal_drawdown_upside_ch...,release16_rulebase_reversal_drawdown_upside_ch...,official_daily_path_pool,bc_initialization,3,1,False,1,False,False,-0.427053,19.326906,0.200377,30.427053,0.000000,-30.427053,0.673094,57.435335,trend_capture
7,release16_rulebase_reversal_drawdown_upside_ch...,release16_rulebase_reversal_drawdown_upside_ch...,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,3,1,False,1,False,False,-0.427053,19.326906,0.200377,30.427053,0.000000,-30.427053,0.673094,57.435335,trend_capture


,fold,required_slice_count,min_required_accuracy,failed_slice_count,slice_precondition_gate
0,wf_2020,4,0.551834,1,False
1,wf_2022,4,0.875000,1,False
2,wf_2023_2024,4,0.363636,2,False


In [3]:
def _window_gate(relative_return: pd.Series, max_dd: pd.Series) -> pd.Series:
    return (pd.to_numeric(relative_return, errors="coerce") >= TARGET_RELATIVE_RETURN_PP) & (
        pd.to_numeric(max_dd, errors="coerce") < TARGET_MAX_DD_PCT
    )


slice_counts = (
    slice_precondition.assign(fold=slice_precondition["fold"].astype(str))
    .set_index("fold")["failed_slice_count"]
    .to_dict()
)

terms = candidate_windows.copy()
terms["candidate_role"] = "official_replay_candidate"
terms["runnable_actor_candidate"] = True
terms["relative_loss"] = (TARGET_RELATIVE_RETURN_PP - pd.to_numeric(terms["relative_return"], errors="coerce")).clip(lower=0.0)
terms["drawdown_gap_pct"] = (pd.to_numeric(terms["max_dd"], errors="coerce") - TARGET_MAX_DD_PCT).clip(lower=0.0)
terms["drawdown_loss"] = terms["drawdown_gap_pct"] * DRAWDOWN_DRIFT_WEIGHT
terms["slice_failed_count"] = terms["window"].map(slice_counts).fillna(0).astype(int)
terms["sparse_slice_guard_loss"] = terms["slice_failed_count"] * SPARSE_SLICE_GUARD_WEIGHT
terms["window_metric_loss"] = terms["relative_loss"] + terms["drawdown_loss"]
terms["objective_window_loss"] = terms["window_metric_loss"] + terms["sparse_slice_guard_loss"]
terms["window_gate_20dd"] = _window_gate(terms["relative_return"], terms["max_dd"])
terms["slice_gate_pass"] = terms["slice_failed_count"].eq(0)
terms["objective_window_pass"] = terms["window_gate_20dd"] & terms["slice_gate_pass"]

teacher = pd.DataFrame(
    {
        "source_label": "release16_nonconvex_checkpoint_objective_readiness_v1",
        "mechanism": "per_fold_teacher_diagnostic",
        "candidate": "per_fold_teacher_mix",
        "candidate_id": "release16_nonconvex_checkpoint_objective_readiness_v1::per_fold_teacher_mix",
        "window": teacher_metrics["window"].astype(str),
        "relative_return": pd.to_numeric(teacher_metrics["relative_return_pct"], errors="coerce"),
        "max_dd": pd.to_numeric(teacher_metrics["max_dd_pct"], errors="coerce"),
        "avg_pos": pd.to_numeric(teacher_metrics["avg_pos"], errors="coerce"),
        "window_gate_pass": _window_gate(teacher_metrics["relative_return_pct"], teacher_metrics["max_dd_pct"]),
        "relative_gap": (TARGET_RELATIVE_RETURN_PP - pd.to_numeric(teacher_metrics["relative_return_pct"], errors="coerce")).clip(lower=0.0),
        "max_dd_gap": (pd.to_numeric(teacher_metrics["max_dd_pct"], errors="coerce") - TARGET_MAX_DD_PCT).clip(lower=0.0),
        "robust_window_gate": False,
        "candidate_role": "diagnostic_per_fold_teacher_mix",
        "runnable_actor_candidate": False,
    }
)
teacher["relative_loss"] = teacher["relative_gap"]
teacher["drawdown_gap_pct"] = teacher["max_dd_gap"]
teacher["drawdown_loss"] = teacher["drawdown_gap_pct"] * DRAWDOWN_DRIFT_WEIGHT
teacher["slice_failed_count"] = teacher["window"].map(slice_counts).fillna(0).astype(int)
teacher["sparse_slice_guard_loss"] = teacher["slice_failed_count"] * SPARSE_SLICE_GUARD_WEIGHT
teacher["window_metric_loss"] = teacher["relative_loss"] + teacher["drawdown_loss"]
teacher["objective_window_loss"] = teacher["window_metric_loss"] + teacher["sparse_slice_guard_loss"]
teacher["window_gate_20dd"] = _window_gate(teacher["relative_return"], teacher["max_dd"])
teacher["slice_gate_pass"] = teacher["slice_failed_count"].eq(0)
teacher["objective_window_pass"] = teacher["window_gate_20dd"] & teacher["slice_gate_pass"]

objective_terms = pd.concat([terms, teacher], ignore_index=True, sort=False)
objective_terms = objective_terms[
    [
        "source_label",
        "mechanism",
        "candidate",
        "candidate_id",
        "candidate_role",
        "runnable_actor_candidate",
        "window",
        "relative_return",
        "max_dd",
        "avg_pos",
        "relative_loss",
        "drawdown_gap_pct",
        "drawdown_loss",
        "slice_failed_count",
        "sparse_slice_guard_loss",
        "window_metric_loss",
        "objective_window_loss",
        "window_gate_20dd",
        "slice_gate_pass",
        "objective_window_pass",
    ]
].sort_values(["candidate_id", "window"])

objective_terms.to_csv(CARD / "objective_window_terms.csv", index=False)
display(objective_terms.head(10))


,source_label,mechanism,candidate,candidate_id,candidate_role,runnable_actor_candidate,window,relative_return,max_dd,avg_pos,relative_loss,drawdown_gap_pct,drawdown_loss,slice_failed_count,sparse_slice_guard_loss,window_metric_loss,objective_window_loss,window_gate_20dd,slice_gate_pass,objective_window_pass
0,cap10_three_fold_checkpoint_metric_replay_v2,official_daily_path_pool,bc_initialization,cap10_three_fold_checkpoint_metric_replay_v2::...,official_replay_candidate,True,wf_2020,-17.513245,14.984701,0.232523,47.513245,0.000000,0.00000,1,5.0,47.513245,52.513245,False,False,False
4,cap10_three_fold_checkpoint_metric_replay_v2,official_daily_path_pool,bc_initialization,cap10_three_fold_checkpoint_metric_replay_v2::...,official_replay_candidate,True,wf_2022,73.879736,18.802284,0.142739,0.000000,0.000000,0.00000,1,5.0,0.000000,5.000000,True,False,False
8,cap10_three_fold_checkpoint_metric_replay_v2,official_daily_path_pool,bc_initialization,cap10_three_fold_checkpoint_metric_replay_v2::...,official_replay_candidate,True,wf_2023_2024,-10.521654,19.933946,0.167156,40.521654,0.000000,0.00000,2,10.0,40.521654,50.521654,False,False,False
1,cap10_three_fold_checkpoint_metric_replay_v2,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,cap10_three_fold_checkpoint_metric_replay_v2::...,official_replay_candidate,True,wf_2020,-17.513245,14.984701,0.232523,47.513245,0.000000,0.00000,1,5.0,47.513245,52.513245,False,False,False
5,cap10_three_fold_checkpoint_metric_replay_v2,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,cap10_three_fold_checkpoint_metric_replay_v2::...,official_replay_candidate,True,wf_2022,73.879736,18.802284,0.142739,0.000000,0.000000,0.00000,1,5.0,0.000000,5.000000,True,False,False
9,cap10_three_fold_checkpoint_metric_replay_v2,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,cap10_three_fold_checkpoint_metric_replay_v2::...,official_replay_candidate,True,wf_2023_2024,-10.521654,19.933946,0.167156,40.521654,0.000000,0.00000,2,10.0,40.521654,50.521654,False,False,False
2,cap10_three_fold_checkpoint_metric_replay_v2,official_daily_path_pool,ppo_retained_iter_4,cap10_three_fold_checkpoint_metric_replay_v2::...,official_replay_candidate,True,wf_2020,3.230479,20.918285,0.360661,26.769521,0.918285,1.83657,1,5.0,28.606091,33.606091,False,False,False
6,cap10_three_fold_checkpoint_metric_replay_v2,official_daily_path_pool,ppo_retained_iter_4,cap10_three_fold_checkpoint_metric_replay_v2::...,official_replay_candidate,True,wf_2022,79.307981,10.123049,0.075017,0.000000,0.000000,0.00000,1,5.0,0.000000,5.000000,True,False,False
10,cap10_three_fold_checkpoint_metric_replay_v2,official_daily_path_pool,ppo_retained_iter_4,cap10_three_fold_checkpoint_metric_replay_v2::...,official_replay_candidate,True,wf_2023_2024,-0.268838,8.955769,0.136020,30.268838,0.000000,0.00000,2,10.0,30.268838,40.268838,False,False,False
3,cap10_three_fold_checkpoint_metric_replay_v2,official_daily_path_pool,ppo_selected_best,cap10_three_fold_checkpoint_metric_replay_v2::...,official_replay_candidate,True,wf_2020,3.230479,20.918285,0.360661,26.769521,0.918285,1.83657,1,5.0,28.606091,33.606091,False,False,False


In [4]:
def score_candidate_group(group: pd.DataFrame) -> dict[str, Any]:
    metric_fail = sorted(group.loc[~group["window_gate_20dd"], "window"].astype(str).unique())
    slice_fail = sorted(group.loc[~group["slice_gate_pass"], "window"].astype(str).unique())
    objective_fail = sorted(group.loc[~group["objective_window_pass"], "window"].astype(str).unique())
    worst = group.sort_values("objective_window_loss", ascending=False).iloc[0]
    return {
        "candidate_id": str(group["candidate_id"].iloc[0]),
        "source_label": str(group["source_label"].iloc[0]),
        "mechanism": str(group["mechanism"].iloc[0]),
        "candidate": str(group["candidate"].iloc[0]),
        "candidate_role": str(group["candidate_role"].iloc[0]),
        "runnable_actor_candidate": bool(group["runnable_actor_candidate"].iloc[0]),
        "window_count": int(len(group)),
        "window_pass_count": int(group["window_gate_20dd"].sum()),
        "slice_pass_count": int(group["slice_gate_pass"].sum()),
        "objective_window_pass_count": int(group["objective_window_pass"].sum()),
        "window_gate": bool(group["window_gate_20dd"].all()),
        "slice_precondition_gate": bool(group["slice_gate_pass"].all()),
        "objective_pass": bool(group["objective_window_pass"].all()),
        "objective_loss": float(group["objective_window_loss"].sum()),
        "window_metric_loss": float(group["window_metric_loss"].sum()),
        "relative_loss": float(group["relative_loss"].sum()),
        "drawdown_loss": float(group["drawdown_loss"].sum()),
        "sparse_slice_guard_loss": float(group["sparse_slice_guard_loss"].sum()),
        "min_relative_return": float(pd.to_numeric(group["relative_return"], errors="coerce").min()),
        "max_max_dd": float(pd.to_numeric(group["max_dd"], errors="coerce").max()),
        "mean_avg_pos": float(pd.to_numeric(group["avg_pos"], errors="coerce").mean()),
        "worst_window": str(worst["window"]),
        "metric_failed_windows": ",".join(metric_fail),
        "slice_failed_windows": ",".join(slice_fail),
        "objective_failed_windows": ",".join(objective_fail),
    }


objective_scores = pd.DataFrame(
    [score_candidate_group(group) for _, group in objective_terms.groupby("candidate_id", sort=True)]
).sort_values(
    ["objective_pass", "objective_loss", "runnable_actor_candidate", "candidate_id"],
    ascending=[False, True, False, True],
).reset_index(drop=True)
objective_scores.insert(0, "objective_rank", np.arange(1, len(objective_scores) + 1))
objective_scores.to_csv(CARD / "objective_candidate_scores.csv", index=False)

best_training_target = objective_scores.iloc[0].to_dict()
best_runnable = objective_scores[objective_scores["runnable_actor_candidate"]].iloc[0].to_dict()
best_runnable_terms = objective_terms[objective_terms["candidate_id"].eq(best_runnable["candidate_id"])]
best_runnable_terms.to_csv(CARD / "best_runnable_window_terms.csv", index=False)

print("best_training_target", best_training_target)
print("best_runnable", best_runnable)
display(objective_scores.head(12))


best_training_target {'objective_rank': 1, 'candidate_id': 'release16_nonconvex_checkpoint_objective_readiness_v1::per_fold_teacher_mix', 'source_label': 'release16_nonconvex_checkpoint_objective_readiness_v1', 'mechanism': 'per_fold_teacher_diagnostic', 'candidate': 'per_fold_teacher_mix', 'candidate_role': 'diagnostic_per_fold_teacher_mix', 'runnable_actor_candidate': False, 'window_count': 3, 'window_pass_count': 1, 'slice_pass_count': 0, 'objective_window_pass_count': 0, 'window_gate': False, 'slice_precondition_gate': False, 'objective_pass': False, 'objective_loss': 35.880907, 'window_metric_loss': 15.880907, 'relative_loss': 15.880907, 'drawdown_loss': 0.0, 'sparse_slice_guard_loss': 20.0, 'min_relative_return': 21.33313, 'max_max_dd': 16.652151, 'mean_avg_pos': 0.16380366666666668, 'worst_window': 'wf_2023_2024', 'metric_failed_windows': 'wf_2020,wf_2023_2024', 'slice_failed_windows': 'wf_2020,wf_2022,wf_2023_2024', 'objective_failed_windows': 'wf_2020,wf_2022,wf_2023_2024'}
be

,objective_rank,candidate_id,source_label,mechanism,candidate,candidate_role,runnable_actor_candidate,window_count,window_pass_count,slice_pass_count,...,relative_loss,drawdown_loss,sparse_slice_guard_loss,min_relative_return,max_max_dd,mean_avg_pos,worst_window,metric_failed_windows,slice_failed_windows,objective_failed_windows
0,1,release16_nonconvex_checkpoint_objective_readi...,release16_nonconvex_checkpoint_objective_readi...,per_fold_teacher_diagnostic,per_fold_teacher_mix,diagnostic_per_fold_teacher_mix,False,3,1,0,...,15.880907,0.000000,20.0,21.333130,16.652151,0.163804,wf_2023_2024,"wf_2020,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024"
1,2,release16_bull_reentry_participation_objective...,release16_bull_reentry_participation_objective...,official_daily_path_pool,bull65_hold3,official_replay_candidate,True,3,1,0,...,23.416299,2.212312,20.0,14.302062,21.106156,0.236935,wf_2023_2024,"wf_2020,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024"
2,3,release16_bull_reentry_participation_objective...,release16_bull_reentry_participation_objective...,official_daily_path_pool,bull70_hold5,official_replay_candidate,True,3,1,0,...,33.748876,1.878259,20.0,11.949062,20.939130,0.210516,wf_2023_2024,"wf_2020,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024"
3,4,release16_bull_reentry_participation_objective...,release16_bull_reentry_participation_objective...,official_daily_path_pool,release64_current_target,official_replay_candidate,True,3,0,0,...,23.416299,15.933441,20.0,14.302062,26.860565,0.255722,wf_2023_2024,"wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024"
4,5,release16_release_gate_bc_official_replay_v1::...,release16_release_gate_bc_official_replay_v1,official_daily_path_pool,release64_no_objective,official_replay_candidate,True,3,0,0,...,23.416299,15.933441,20.0,14.302062,26.860565,0.255722,wf_2023_2024,"wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024"
5,6,release16_failedfold_overlay_checkpoint_screen...,release16_failedfold_overlay_checkpoint_screen_v1,official_daily_path_pool,bc_initialization,official_replay_candidate,True,3,1,0,...,45.173038,0.000000,20.0,3.406380,16.652151,0.131609,wf_2023_2024,"wf_2020,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024"
6,7,release16_failedfold_overlay_checkpoint_screen...,release16_failedfold_overlay_checkpoint_screen_v1,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,official_replay_candidate,True,3,1,0,...,45.173038,0.000000,20.0,3.406380,16.652151,0.131609,wf_2023_2024,"wf_2020,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024"
7,8,release16_rulebase_reversal_checkpoint_metric_...,release16_rulebase_reversal_checkpoint_metric_...,official_daily_path_pool,bc_initialization,official_replay_candidate,True,3,1,0,...,43.338766,1.869225,20.0,5.082695,20.934612,0.174185,wf_2020,"wf_2020,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024"
8,9,release16_rulebase_reversal_checkpoint_metric_...,release16_rulebase_reversal_checkpoint_metric_...,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,official_replay_candidate,True,3,1,0,...,43.338766,1.869225,20.0,5.082695,20.934612,0.174185,wf_2020,"wf_2020,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024"
9,10,release16_release_gate_bc_official_replay_v1::...,release16_release_gate_bc_official_replay_v1,official_daily_path_pool,baseline_current,official_replay_candidate,True,3,0,0,...,36.911999,10.470697,20.0,7.186027,24.834087,0.238562,wf_2023_2024,"wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2022,wf_2023_2024"


In [5]:
# Build a compact handoff. The helper supports objective_pass/objective_loss tables,
# and the cluster frame keeps the underparticipation failure explicit for the next training loop.
poor_window_clusters = pd.DataFrame(
    [
        {
            "cluster_id": "wf_2020_relative_floor_and_drawdown_drift",
            "dominant_failure_type": "underparticipation",
            "cluster_start": "2020-01-02",
            "cluster_end": "2020-12-30",
            "mean_excess_return": -float(best_runnable.get("relative_loss", 0.0)),
            "mean_avg_pos": float(best_runnable.get("mean_avg_pos", 0.0)),
            "source_card": "release16_expanded_official_candidate_pool_v1",
        },
        {
            "cluster_id": "wf_2023_2024_relative_floor_gap",
            "dominant_failure_type": "underparticipation",
            "cluster_start": "2024-01-02",
            "cluster_end": "2024-12-30",
            "mean_excess_return": -float(best_training_target.get("relative_loss", 0.0)),
            "mean_avg_pos": float(best_training_target.get("mean_avg_pos", 0.0)),
            "source_card": "release16_expanded_official_candidate_pool_v1",
        },
    ]
)
poor_window_clusters.to_csv(CARD / "poor_window_clusters.csv", index=False)

handoff = derive_training_objective_handoff(
    walkforward_scores=objective_scores.rename(columns={"candidate_id": "candidate"}),
    poor_window_clusters=poor_window_clusters,
    feature_groups=pd.DataFrame(
        [
            {"feature_group": "state_head_sparse_release_slice", "top_mentions": 3, "mean_abs_delta_z": 1.0},
            {"feature_group": "relative_strength_spread", "top_mentions": 2, "mean_abs_delta_z": 0.7},
        ]
    ),
)
handoff["decision"] = "train_nonconvex_checkpoint_objective_with_sparse_slice_guard"
handoff["promotion"] = "blocked"
handoff["selected_diagnostic_candidate"] = str(best_training_target["candidate_id"])
handoff["selected_runnable_reference_candidate"] = str(best_runnable["candidate_id"])
handoff["objective_terms"] = [
    "official_replay_relative_floor_loss",
    "official_replay_drawdown_drift_stop_loss",
    "state_head_sparse_slice_fit_guard",
    "fold_local_teacher_alignment_loss",
]
handoff["stop_mechanisms"] = [
    "path_level_threshold_router_search",
    "actor_retention_before_official_three_fold_replay",
]
handoff["next_action"] = "train_fold_local_nonconvex_checkpoint_actor_with_relative_floor_drawdown_drift_stop_and_sparse_slice_guard_then_official_three_fold_replay"

(CARD / "training_objective_handoff.json").write_text(
    json.dumps(json_safe(handoff), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
(CARD / "training_objective_handoff.md").write_text(
    render_training_objective_handoff(handoff),
    encoding="utf-8",
)

source_manifest = pd.DataFrame(
    [
        {
            "source_card": "release16_expanded_official_candidate_pool_v1",
            "source_file": str(SOURCE_PATHS["expanded_candidate_windows"].relative_to(ROOT)),
            "purpose": "official replay candidate pool scored into objective terms",
        },
        {
            "source_card": "release16_nonconvex_checkpoint_objective_readiness_v1",
            "source_file": str(SOURCE_PATHS["readiness_teacher"].relative_to(ROOT)),
            "purpose": "per-fold teacher mix and sparse slice guard reference",
        },
    ]
)
source_manifest.to_csv(CARD / "source_manifest.csv", index=False)

display(Markdown(render_training_objective_handoff(handoff)))


# Training Objective Handoff

## Decision

- decision: `train_nonconvex_checkpoint_objective_with_sparse_slice_guard`
- promotion: `blocked`
- selected_diagnostic_candidate: `release16_nonconvex_checkpoint_objective_readiness_v1::per_fold_teacher_mix`

## Objective Terms

- `official_replay_relative_floor_loss`
- `official_replay_drawdown_drift_stop_loss`
- `state_head_sparse_slice_fit_guard`
- `fold_local_teacher_alignment_loss`

## Risk Gap

- max_dd_gap=1.65pp, relative_gap=8.67pp, worst_window=`wf_2023_2024`

## Primary Underparticipation Cluster

- cluster=`wf_2020_relative_floor_and_drawdown_drift`, start=2020-01-02, end=2020-12-30, mean_excess=-23.42pp

## Stop Mechanisms

- `path_level_threshold_router_search`
- `actor_retention_before_official_three_fold_replay`


In [6]:
selected_by_fold: dict[str, dict[str, Any]] = {}
for row in objective_terms[objective_terms["candidate_id"].eq(best_training_target["candidate_id"])].itertuples(index=False):
    selected_by_fold[str(row.window)] = {
        "selected_candidate": str(row.candidate),
        "selected_role": str(row.candidate_role),
        "selected_relative_return": float(row.relative_return),
        "selected_max_dd": float(row.max_dd),
        "selected_avg_pos": float(row.avg_pos),
        "relative_gap_to_target_pp": float(row.relative_loss),
        "drawdown_gap_to_target_pp": float(row.drawdown_gap_pct),
        "sparse_slice_failed_count": int(row.slice_failed_count),
        "selected_candidate_gate": bool(row.window_gate_20dd),
        "slice_gate": bool(row.slice_gate_pass),
        "objective_window_pass": bool(row.objective_window_pass),
    }

metrics = [
    {
        "window": window,
        "split_type": "walk_forward_test",
        "relative_return_pct": payload["selected_relative_return"],
        "max_dd_pct": payload["selected_max_dd"],
        "avg_pos": payload["selected_avg_pos"],
    }
    for window, payload in selected_by_fold.items()
]

objective_contract = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_card": "release16_expanded_official_candidate_pool_v1",
    "direct_comparison_baseline": "release16_expanded_official_candidate_pool_v1",
    "readiness_reference": "release16_nonconvex_checkpoint_objective_readiness_v1",
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct_strict_lt": TARGET_MAX_DD_PCT,
    "drawdown_drift_weight": DRAWDOWN_DRIFT_WEIGHT,
    "sparse_slice_guard_weight": SPARSE_SLICE_GUARD_WEIGHT,
    "required_windows": REQUIRED_WINDOWS,
    "best_training_target_candidate_id": best_training_target["candidate_id"],
    "best_runnable_reference_candidate_id": best_runnable["candidate_id"],
    "objective_pass_candidate_count": int(objective_scores["objective_pass"].sum()),
    "runnable_objective_pass_candidate_count": int(
        (objective_scores["objective_pass"] & objective_scores["runnable_actor_candidate"]).sum()
    ),
    "actor_retention_before_replay": False,
    "source_paths": [str(path.relative_to(ROOT)) for path in SOURCE_PATHS.values()],
    "required_next_action": handoff["next_action"],
}
(CARD / "objective_contract.json").write_text(
    json.dumps(json_safe(objective_contract), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

summary_points = [
    "Expanded official replay pool still has zero objective pass candidates once relative floor, 20% drawdown, and sparse slice guards are applied.",
    f"Best runnable reference remains {best_runnable['candidate_id']} with min relative {best_runnable['min_relative_return']:.2f}pp and worst MaxDD {best_runnable['max_max_dd']:.2f}%.",
    f"Best diagnostic training target is {best_training_target['candidate_id']} with objective loss {best_training_target['objective_loss']:.2f}; it is not a retained actor.",
    "Sparse state-head slice precondition fails on every required fold, so PPO/actor retention is blocked until the slice guard is satisfied and official replay passes.",
    "Next action is fold-local nonconvex checkpoint training with relative floor, drawdown drift stop, and sparse slice guard, followed by official three-fold replay.",
]

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "nonconvex_checkpoint_objective_scored_training_contract_no_actor_retention",
    "role": "diagnose",
    "changed_layer": ["model_selection", "checkpoint_objective", "state_head_slice_guard"],
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": "release16_expanded_official_candidate_pool_v1",
    "source_card": "release16_expanded_official_candidate_pool_v1",
    "source_path": "runtime/model_selection/card/release16_expanded_official_candidate_pool_v1/summary.json",
    "source_paths": objective_contract["source_paths"],
    "readiness_reference": "release16_nonconvex_checkpoint_objective_readiness_v1",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_from_this_card": True,
    "next_training_allowed_after_target_revision": False,
    "target_contract_gate": True,
    "bc_readiness_gate": bool(readiness_summary.get("bc_readiness_gate", False)),
    "bc_screen_allowed_next": True,
    "ppo_allowed_next": False,
    "state_head_slice_precondition_gate": bool(slice_precondition["slice_precondition_gate"].astype(bool).all()),
    "objective_pass_candidate_count": objective_contract["objective_pass_candidate_count"],
    "runnable_objective_pass_candidate_count": objective_contract["runnable_objective_pass_candidate_count"],
    "official_pool_candidate_count": int(official_scores["candidate_id"].nunique()),
    "official_pool_window_count": int(len(candidate_windows)),
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_relative_return_by_window_pp": {window: TARGET_RELATIVE_RETURN_PP for window in REQUIRED_WINDOWS},
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "drawdown_gate_operator": "<",
    "target_window": REQUIRED_WINDOWS,
    "metric_basis": "diagnostic_per_fold_teacher_mix_not_retained_actor_objective_target",
    "metrics": metrics,
    "selected_by_fold": selected_by_fold,
    "best_training_target": json_safe(best_training_target),
    "best_runnable_reference": json_safe(best_runnable),
    "min_wf_relative_return_pp": float(min(item["relative_return_pct"] for item in metrics)),
    "worst_wf_max_drawdown_pct": float(max(item["max_dd_pct"] for item in metrics)),
    "max_relative_gap_to_target_pp": float(max(payload["relative_gap_to_target_pp"] for payload in selected_by_fold.values())),
    "max_drawdown_gap_to_target_pp": float(max(payload["drawdown_gap_to_target_pp"] for payload in selected_by_fold.values())),
    "failed_folds": [
        window for window, payload in selected_by_fold.items() if not payload["objective_window_pass"]
    ],
    "primary_blocker": "training_contract_ready",
    "next_mechanism": handoff["next_action"],
    "training_objective_handoff": json_safe(handoff),
    "objective_terms": objective_contract,
    "summary_points": summary_points,
    "sidecars": {
        "objective_candidate_scores": str((CARD / "objective_candidate_scores.csv").relative_to(ROOT)),
        "objective_window_terms": str((CARD / "objective_window_terms.csv").relative_to(ROOT)),
        "best_runnable_window_terms": str((CARD / "best_runnable_window_terms.csv").relative_to(ROOT)),
        "poor_window_clusters": str((CARD / "poor_window_clusters.csv").relative_to(ROOT)),
        "source_manifest": str((CARD / "source_manifest.csv").relative_to(ROOT)),
        "objective_contract": str((CARD / "objective_contract.json").relative_to(ROOT)),
        "training_objective_handoff": str((CARD / "training_objective_handoff.json").relative_to(ROOT)),
        "selection": str((CARD / "selection.json").relative_to(ROOT)),
        "summary": str((CARD / "summary.json").relative_to(ROOT)),
        "conclusion": str((CARD / "conclusion.md").relative_to(ROOT)),
    },
}

selection = {
    "experiment": EXPERIMENT,
    "decision": summary["decision"],
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "source_card": summary["source_card"],
    "direct_comparison_baseline": summary["direct_comparison_baseline"],
    "best_training_target_candidate_id": best_training_target["candidate_id"],
    "best_runnable_reference_candidate_id": best_runnable["candidate_id"],
    "next_mechanism": summary["next_mechanism"],
    "objective_pass_candidate_count": summary["objective_pass_candidate_count"],
    "runnable_objective_pass_candidate_count": summary["runnable_objective_pass_candidate_count"],
    "target_contract_gate": summary["target_contract_gate"],
    "bc_readiness_gate": summary["bc_readiness_gate"],
    "bc_screen_allowed_next": summary["bc_screen_allowed_next"],
    "state_head_slice_precondition_gate": summary["state_head_slice_precondition_gate"],
    "source_paths": summary["source_paths"],
}

(CARD / "summary.json").write_text(json.dumps(json_safe(summary), indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "selection.json").write_text(json.dumps(json_safe(selection), indent=2, sort_keys=True) + "\n", encoding="utf-8")

conclusion = "\n".join(
    [
        f"# {EXPERIMENT} Conclusion",
        "",
        "## Decision",
        "",
        f"- decision: `{summary['decision']}`",
        "- promotion_allowed: `False`",
        "- actor_retention_allowed: `False`",
        f"- next_mechanism: `{summary['next_mechanism']}`",
        "",
        "## Key Metrics",
        "",
        f"- objective_pass_candidate_count: `{summary['objective_pass_candidate_count']}`; runnable_objective_pass_candidate_count: `{summary['runnable_objective_pass_candidate_count']}`.",
        f"- best runnable reference: `{best_runnable['candidate_id']}` with min relative `{best_runnable['min_relative_return']:.2f}pp`, max MaxDD `{best_runnable['max_max_dd']:.2f}%`, objective loss `{best_runnable['objective_loss']:.2f}`.",
        f"- best diagnostic training target: `{best_training_target['candidate_id']}` with min relative `{best_training_target['min_relative_return']:.2f}pp`, max MaxDD `{best_training_target['max_max_dd']:.2f}%`, objective loss `{best_training_target['objective_loss']:.2f}`.",
        f"- sparse slice gate: `{summary['state_head_slice_precondition_gate']}`; failed folds: `{','.join(summary['failed_folds'])}`.",
        "",
        "## Risk Notes",
        "",
        "- The best diagnostic row is a per-fold teacher mix, not a replayable actor. It cannot be promoted or retained.",
        "- The best runnable row still fails relative-return floors and breaches the 20% drawdown gate in wf_2020.",
        "- The next run must train against this objective and then pass official wf_2020/wf_2022/wf_2023_2024 replay before retention.",
        "",
        "## Artifacts",
        "",
        "- `objective_candidate_scores.csv`",
        "- `objective_window_terms.csv`",
        "- `training_objective_handoff.json` / `training_objective_handoff.md`",
        "- `objective_contract.json`",
        "- `summary.json` / `selection.json`",
    ]
) + "\n"
(CARD / "conclusion.md").write_text(conclusion, encoding="utf-8")

display(Markdown(conclusion))
print("wrote", CARD)


# release16_nonconvex_checkpoint_objective_score_handoff_v1 Conclusion

## Decision

- decision: `nonconvex_checkpoint_objective_scored_training_contract_no_actor_retention`
- promotion_allowed: `False`
- actor_retention_allowed: `False`
- next_mechanism: `train_fold_local_nonconvex_checkpoint_actor_with_relative_floor_drawdown_drift_stop_and_sparse_slice_guard_then_official_three_fold_replay`

## Key Metrics

- objective_pass_candidate_count: `0`; runnable_objective_pass_candidate_count: `0`.
- best runnable reference: `release16_bull_reentry_participation_objective_replay_v1::bull65_hold3` with min relative `14.30pp`, max MaxDD `21.11%`, objective loss `45.63`.
- best diagnostic training target: `release16_nonconvex_checkpoint_objective_readiness_v1::per_fold_teacher_mix` with min relative `21.33pp`, max MaxDD `16.65%`, objective loss `35.88`.
- sparse slice gate: `False`; failed folds: `wf_2020,wf_2022,wf_2023_2024`.

## Risk Notes

- The best diagnostic row is a per-fold teacher mix, not a replayable actor. It cannot be promoted or retained.
- The best runnable row still fails relative-return floors and breaches the 20% drawdown gate in wf_2020.
- The next run must train against this objective and then pass official wf_2020/wf_2022/wf_2023_2024 replay before retention.

## Artifacts

- `objective_candidate_scores.csv`
- `objective_window_terms.csv`
- `training_objective_handoff.json` / `training_objective_handoff.md`
- `objective_contract.json`
- `summary.json` / `selection.json`


wrote /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_nonconvex_checkpoint_objective_score_handoff_v1
